# Department Top Salaries

**Project type:** DataFrame creation, grouping, deduplication, and merging  
**Tools:** Python, pandas

## Goal
Return employees whose salary belongs to one of the **three highest distinct salary levels in their department**. If multiple employees have the same qualifying salary, every tied employee is retained.

> **Interpretation note:** The assignment handout's prose and example output are not fully consistent. This notebook follows the top-three-distinct-salary-level interpretation suggested by its `drop_duplicates`, grouping, and `head` requirements. If a department has fewer than three distinct salary levels, all available levels qualify.

## 1. Create the sample DataFrames

The exercise provides these small example records directly, so there are no external data files for this project.

In [1]:
import pandas as pd

employee_records = [
    [1, "Joe", 85000, 1],
    [2, "Henry", 80000, 2],
    [3, "Sam", 60000, 2],
    [4, "Max", 90000, 1],
    [5, "Janet", 69000, 1],
    [6, "Randy", 85000, 1],
    [7, "Will", 70000, 1],
]

department_records = [
    [1, "IT"],
    [2, "Sales"],
]

employees = pd.DataFrame(
    employee_records,
    columns=["EmployeeID", "Employee", "Salary", "Department_ID"],
)

departments = pd.DataFrame(
    department_records,
    columns=["Department_ID", "Department"],
)

display(employees)
display(departments)

,EmployeeID,Employee,Salary,Department_ID
0,1,Joe,85000,1
1,2,Henry,80000,2
2,3,Sam,60000,2
3,4,Max,90000,1
4,5,Janet,69000,1
5,6,Randy,85000,1
6,7,Will,70000,1


,Department_ID,Department
0,1,IT
1,2,Sales


## 2. Join employees to departments

In [2]:
employees_by_department = employees.merge(
    departments,
    on="Department_ID",
    how="inner",
    validate="many_to_one",
)

display(employees_by_department[["Department", "Employee", "Salary"]])

,Department,Employee,Salary
0,IT,Joe,85000
1,Sales,Henry,80000
2,Sales,Sam,60000
3,IT,Max,90000
4,IT,Janet,69000
5,IT,Randy,85000
6,IT,Will,70000


## 3. Find the three highest distinct salary levels in each department

First keep unique department/salary pairs, sort salaries from highest to lowest inside each department, and select up to three distinct salary levels. This avoids counting a tied salary more than once when choosing the three levels.

In [3]:
unique_salary_levels = (
    employees_by_department[["Department", "Salary"]]
    .drop_duplicates()
    .sort_values(["Department", "Salary"], ascending=[True, False])
)

top_salary_levels = (
    unique_salary_levels.groupby("Department", group_keys=False)
    .head(3)
)

display(top_salary_levels)

,Department,Salary
3,IT,90000
0,IT,85000
6,IT,70000
1,Sales,80000
2,Sales,60000


## 4. Return all employees at those salary levels

Merge the qualifying salary levels back to the employee table. Because the merge is on both department and salary, all employees tied at a selected level are included.

In [4]:
result = employees_by_department.merge(
    top_salary_levels,
    on=["Department", "Salary"],
    how="inner",
    validate="many_to_one",
)

result = (
    result[["Department", "Employee", "Salary"]]
    .sort_values(["Department", "Salary", "Employee"], ascending=[True, False, True])
    .reset_index(drop=True)
)

display(result)

,Department,Employee,Salary
0,IT,Max,90000
1,IT,Joe,85000
2,IT,Randy,85000
3,IT,Will,70000
4,Sales,Henry,80000
5,Sales,Sam,60000


## 5. Simple checks

These checks ensure the qualifying salary levels are unique by department and tied employees are not lost in the final merge.

In [5]:
assert not top_salary_levels.duplicated(["Department", "Salary"]).any()
assert {"Joe", "Randy"}.issubset(set(result["Employee"]))
assert ((result["Department"] == "IT") & (result["Salary"] == 85000)).sum() == 2

print("Checks passed: unique salary levels are selected and tied employees are retained.")

Checks passed: unique salary levels are selected and tied employees are retained.


## Interpretation and limitations

This is a small educational example that demonstrates DataFrame construction, `merge`, `drop_duplicates`, grouping, sorting, `head`, and selecting output columns. It is not real HR data. In a department with fewer than three distinct salary levels, all available levels appear in the result.